In [ ]:
import math
class Value:

    def __init__(self,data,_children=(),_op='',label=''):
        self.data=data
        self._prev=set(_children)
        self._backward = lambda: None
        self.grad=0
        self._op=_op
        self.label=label

    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self,other):
        other = other if isinstance(other, Value) else Value(other)
    #out = Value(self.data + other.data, (self, other), '+')
        out=Value(self.data+other.data,(self,other),'+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward

        return out

    def __sub__(self, other): # self - other
        return self + (-other)
    
    def __mul__(self,other):
        # out=Value(self.data*other.data,(self,other),'*')
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): # other * self
        return self * other

    def __radd__(self, other): # other * self
        return self + other
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
    
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

a=Value(2.0,label='a')
b=Value(-3.0,label='b')
c=Value(10.0,label='c')
a.__add__(b)

e=a*b;e.label='e'
d=e+c;d.label='d'

f=Value(-2.0,label='f')
L=f*d;L.label='L'

In [ ]:
import random

w=[Value(random.uniform(-1,1)) for _ in range(4)]
w

In [ ]:
x = [2.0, 3.0, -1.0]
for wi, xi in zip(w, x):
    print(wi,"---",xi,"---",wi*xi)
act = sum((wi*xi for wi, xi in zip(w, x)))
act